# Accuracy Evaluation: LLM Blueprint Analysis

This notebook evaluates how accurately the LLM extracted factual information from construction blueprints by cross-referencing its output against the ground-truth JSON produced during Task 1.

**What it measures:**
1. **Value Presence Rate (VPR):** For properties tagged Vector/OCR, what % of stated values are verifiable in the JSON?
2. **Source Attribution Accuracy (SAA):** When the LLM claims a specific source layer, is it correct?
3. **Hallucination Detection:** Values tagged Vector/OCR but not found in any JSON layer.

---

In [ ]:
# @title 1. Install Dependencies & Imports
!pip install fuzzywuzzy python-Levenshtein -q

import os
import re
import json
import csv
from pathlib import Path
from fuzzywuzzy import fuzz
from collections import defaultdict

print('\u2705 Dependencies installed and imported.')

In [ ]:
# Option: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Setup

Point the paths below to:
1. **`MD_OUTPUT_PATH`**: The `.md` file produced by Task 3 or Task 4 (the LLM analysis output).
2. **`JSON_DIR`**: The `vector_data/` folder produced by Task 1, containing one `.json` file per sheet.
3. **`PROJECT_LABEL`**: A short name for this evaluation run (used in the exported CSV filename).

Upload both the `.md` file and the `vector_data/` folder to Colab, or mount Google Drive.

In [ ]:
# @title 2. Configuration - SET THESE PATHS

# === SET THESE ===
MD_OUTPUT_PATH = ''
JSON_DIR       = ''
PROJECT_LABEL  = ''

# Matching thresholds (tune if needed)
TOKEN_COVERAGE_THRESHOLD = 0.70
FUZZY_MATCH_THRESHOLD    = 80

# Validate paths
assert os.path.isfile(MD_OUTPUT_PATH), f'MD file not found: {MD_OUTPUT_PATH}'
assert os.path.isdir(JSON_DIR), f'JSON directory not found: {JSON_DIR}'

json_files = sorted([f for f in os.listdir(JSON_DIR) if f.endswith('.json')])
print(f'MD file found: {os.path.basename(MD_OUTPUT_PATH)}')
print(f'JSON directory: {len(json_files)} JSON files found')
for jf in json_files:
    print(f'   - {jf}')

In [ ]:
# @title 3. Parse LLM Output (.md) - Extract Summary Tables

def parse_md_output(md_path):
    with open(md_path, 'r', encoding='utf-8') as f:
        content = f.read()

    sheet_blocks = re.split(r'(?=^# ANALYSIS:)', content, flags=re.MULTILINE)
    parsed_sheets = []

    for block in sheet_blocks:
        block = block.strip()
        if not block:
            continue

        header_match = re.match(r'^# ANALYSIS:\s*(.+)', block)
        if not header_match:
            continue

        sheet_name = header_match.group(1).strip()
        sheet_file_stem = sheet_name.replace('.png', '').strip()

        lines = block.split('\n')
        table_rows = []
        in_table = False
        separator_passed = False

        for line in lines:
            stripped = line.strip()

            # Detect the header row of the summary table
            if ('|' in stripped and
                'Property' in stripped and
                ('Value' in stripped or 'Extracted' in stripped) and
                'Source' in stripped):
                in_table = True
                separator_passed = False
                continue

            # Skip separator row
            if in_table and not separator_passed:
                if '---' in stripped or ':---' in stripped:
                    separator_passed = True
                    continue

            # Parse data rows
            if in_table and separator_passed and stripped.startswith('|'):
                cells = [c.strip() for c in stripped.split('|')]
                cells = [c for c in cells if c != '']

                if len(cells) >= 3:
                    table_rows.append({
                        'property': cells[0].strip('* '),
                        'value': cells[1].strip('* '),
                        'source': cells[2].strip('* ')
                    })

            elif in_table and separator_passed and not stripped.startswith('|') and stripped != '':
                in_table = False
                separator_passed = False

        parsed_sheets.append({
            'sheet_name': sheet_name,
            'sheet_file_stem': sheet_file_stem,
            'properties': table_rows
        })

    return parsed_sheets


parsed = parse_md_output(MD_OUTPUT_PATH)
total_props = sum(len(s['properties']) for s in parsed)
sheets_with_tables = [s for s in parsed if len(s['properties']) > 0]

print(f'\nParsed {len(parsed)} sheet analyses from the MD file.')
print(f'{len(sheets_with_tables)} sheets have summary tables ({total_props} total properties).\n')

for s in parsed:
    table_status = f'{len(s["properties"])} properties' if s['properties'] else 'NO TABLE FOUND'
    print(f'   Sheet: {s["sheet_name"]}  ->  {table_status}')

In [ ]:
# @title 4. Load JSON Ground Truth

def load_json_ground_truth(json_dir, parsed_sheets):
    json_files = {Path(f).stem: f for f in os.listdir(json_dir) if f.endswith('.json')}
    mapped = {}
    unmatched = []

    for sheet in parsed_sheets:
        stem = sheet['sheet_file_stem']

        if stem in json_files:
            with open(os.path.join(json_dir, json_files[stem]), 'r') as f:
                mapped[stem] = json.load(f)
            continue

        sheet_num_match = re.search(r'Sheet_(\d+)', stem)
        found = False
        if sheet_num_match:
            sheet_num = sheet_num_match.group(1)
            for jf_stem, jf_name in json_files.items():
                if f'Sheet_{sheet_num}' in jf_stem or f'Sheet_{int(sheet_num):02d}' in jf_stem:
                    with open(os.path.join(json_dir, jf_name), 'r') as f:
                        mapped[stem] = json.load(f)
                    found = True
                    break

        if not found:
            unmatched.append(stem)

    return mapped, unmatched


def extract_texts_from_json(json_data, layer):
    texts = []
    if layer in ('vector', 'all'):
        for item in json_data.get('text', []):
            if isinstance(item, dict) and 'content' in item:
                texts.append(item['content'])
            elif isinstance(item, str):
                texts.append(item)
    if layer in ('ocr', 'all'):
        for item in json_data.get('raster_ocr', []):
            if isinstance(item, dict) and 'content' in item:
                texts.append(item['content'])
            elif isinstance(item, str):
                texts.append(item)
    return texts


json_ground_truth, unmatched = load_json_ground_truth(JSON_DIR, parsed)

print(f'Loaded JSON ground truth for {len(json_ground_truth)} sheets.')
if unmatched:
    print(f'Could not find JSON for: {unmatched}')

# Show vector/OCR text counts for ALL sheets
print()
for stem, jdata in sorted(json_ground_truth.items()):
    v = len(jdata.get('text', []))
    o = len(jdata.get('raster_ocr', []))
    print(f'   {stem}: {v} vector, {o} OCR')


In [ ]:
# @title 5. Matching & Verification Functions

def normalize(text):
    text = str(text).lower().strip()
    text = re.sub(r'\s+', ' ', text)
    # Remove various quote characters
    text = text.replace('"', '').replace("'", '')
    text = text.replace('\u201c', '').replace('\u201d', '')
    text = text.replace('\u2018', '').replace('\u2019', '')
    return text


def tokenize(text):
    normalized = normalize(text)
    tokens = re.findall(r'[\w.+/@#-]+', normalized)
    tokens = [t for t in tokens if len(t) > 1 or t.isdigit()]
    return tokens


def compute_token_coverage(value, corpus_texts):
    value_tokens = tokenize(value)
    if not value_tokens:
        return 0.0, 0, 0
    corpus_joined = ' '.join([normalize(t) for t in corpus_texts])
    found = sum(1 for token in value_tokens if token in corpus_joined)
    coverage = found / len(value_tokens)
    return coverage, found, len(value_tokens)


def fuzzy_match_any(value, corpus_texts, threshold=80):
    norm_value = normalize(value)
    if len(norm_value) < 2:
        return False, 0
    best_score = 0
    for text in corpus_texts:
        norm_text = normalize(text)
        score = max(
            fuzz.partial_ratio(norm_value, norm_text),
            fuzz.token_set_ratio(norm_value, norm_text)
        )
        best_score = max(best_score, score)
        if score >= threshold:
            return True, score
    return False, best_score


def classify_source(source_str):
    s = source_str.lower().strip()
    if 'not found' in s or 'not_found' in s or s == '[not found]':
        return 'not_found'
    has_vector = 'vector' in s
    has_ocr = 'ocr' in s
    has_visual = 'visual' in s
    if has_vector and has_ocr and not has_visual:
        return 'vector+ocr'
    elif has_visual and has_ocr:
        return 'visual+ocr'
    elif has_visual and has_vector:
        return 'visual+vector'
    elif has_vector:
        return 'vector'
    elif has_ocr:
        return 'ocr'
    elif has_visual:
        return 'visual'
    else:
        return 'unknown'


def verify_property(prop, json_data, token_threshold, fuzzy_threshold):
    source_type = classify_source(prop['source'])
    value = prop['value']

    result = {
        'property': prop['property'],
        'value': value,
        'source_claimed': prop['source'],
        'source_type': source_type,
        'verifiable': False,
        'verified': False,
        'found_in_vector': False,
        'found_in_ocr': False,
        'source_correct': None,
        'token_coverage': 0.0,
        'fuzzy_score': 0,
        'method': None,
        'notes': ''
    }

    if source_type == 'not_found':
        result['notes'] = 'Skipped - LLM reported NOT FOUND'
        return result
    if source_type == 'visual':
        result['notes'] = 'Manual review needed - Visual-only source'
        return result
    if source_type == 'unknown':
        result['notes'] = f'Unrecognized source tag: {prop["source"]}'
        return result

    result['verifiable'] = True

    vector_texts = extract_texts_from_json(json_data, 'vector')
    ocr_texts = extract_texts_from_json(json_data, 'ocr')
    all_texts = vector_texts + ocr_texts

    # Check Vector layer
    v_coverage, _, _ = compute_token_coverage(value, vector_texts)
    v_fuzzy, _ = fuzzy_match_any(value, vector_texts, fuzzy_threshold)
    if v_coverage >= token_threshold or v_fuzzy:
        result['found_in_vector'] = True

    # Check OCR layer
    o_coverage, _, _ = compute_token_coverage(value, ocr_texts)
    o_fuzzy, _ = fuzzy_match_any(value, ocr_texts, fuzzy_threshold)
    if o_coverage >= token_threshold or o_fuzzy:
        result['found_in_ocr'] = True

    # Check combined
    a_coverage, _, _ = compute_token_coverage(value, all_texts)
    a_fuzzy, a_fscore = fuzzy_match_any(value, all_texts, fuzzy_threshold)

    if a_coverage >= token_threshold or a_fuzzy:
        result['verified'] = True
        result['method'] = 'token_coverage' if a_coverage >= token_threshold else 'fuzzy_match'

    result['token_coverage'] = round(a_coverage, 3)
    result['fuzzy_score'] = a_fscore

    # Source attribution accuracy
    if source_type == 'vector':
        result['source_correct'] = result['found_in_vector']
    elif source_type == 'ocr':
        result['source_correct'] = result['found_in_ocr']
    elif source_type in ('vector+ocr', 'visual+ocr', 'visual+vector'):
        correct = False
        if 'vector' in source_type and result['found_in_vector']:
            correct = True
        if 'ocr' in source_type and result['found_in_ocr']:
            correct = True
        result['source_correct'] = correct

    # Notes
    if result['verified'] and result['source_correct']:
        result['notes'] = 'Verified - correct source'
    elif result['verified'] and not result['source_correct']:
        actual = []
        if result['found_in_vector']: actual.append('Vector')
        if result['found_in_ocr']: actual.append('OCR')
        lbl = '/'.join(actual) if actual else 'unknown layer'
        result['notes'] = f'Value exists but in {lbl} - source misattributed'
    elif not result['verified']:
        result['notes'] = f'Not found in JSON (coverage: {a_coverage:.0%}, fuzzy: {a_fscore})'

    return result


print('Matching functions loaded.')

In [ ]:
# @title 6. Run Evaluation

all_results = []
sheet_summaries = []

for sheet in parsed:
    stem = sheet['sheet_file_stem']
    props = sheet['properties']

    if not props:
        print(f'SKIP {sheet["sheet_name"]}: No summary table.')
        continue

    if stem not in json_ground_truth:
        print(f'SKIP {sheet["sheet_name"]}: No matching JSON.')
        continue

    json_data = json_ground_truth[stem]
    sheet_results = []

    for prop in props:
        result = verify_property(prop, json_data, TOKEN_COVERAGE_THRESHOLD, FUZZY_MATCH_THRESHOLD)
        result['sheet'] = sheet['sheet_name']
        sheet_results.append(result)
        all_results.append(result)

    verifiable = [r for r in sheet_results if r['verifiable']]
    verified = [r for r in verifiable if r['verified']]
    src_correct = [r for r in verifiable if r['source_correct']]
    visual_only = [r for r in sheet_results if r['source_type'] == 'visual']
    not_found = [r for r in sheet_results if r['source_type'] == 'not_found']
    hallucinated = [r for r in verifiable if not r['verified']]

    vpr = len(verified) / len(verifiable) * 100 if verifiable else 0
    saa = len(src_correct) / len(verifiable) * 100 if verifiable else 0

    sheet_summaries.append({
        'sheet': sheet['sheet_name'],
        'total_properties': len(sheet_results),
        'verifiable': len(verifiable),
        'verified': len(verified),
        'vpr': round(vpr, 1),
        'source_correct': len(src_correct),
        'saa': round(saa, 1),
        'visual_only': len(visual_only),
        'not_found': len(not_found),
        'hallucinated': len(hallucinated)
    })

    status = f'VPR: {vpr:.0f}%  SAA: {saa:.0f}%' if verifiable else 'No verifiable properties'
    print(f'{sheet["sheet_name"]}: {len(props)} props, {len(verifiable)} verifiable -> {status}')

print(f'\nEvaluation complete: {len(all_results)} properties across {len(sheet_summaries)} sheets.')

In [ ]:
# @title 7. Aggregate Results & Display

all_verifiable = [r for r in all_results if r['verifiable']]
all_verified = [r for r in all_verifiable if r['verified']]
all_src_correct = [r for r in all_verifiable if r['source_correct']]
all_visual = [r for r in all_results if r['source_type'] == 'visual']
all_not_found = [r for r in all_results if r['source_type'] == 'not_found']
all_hallucinated = [r for r in all_verifiable if not r['verified']]

agg_vpr = len(all_verified) / len(all_verifiable) * 100 if all_verifiable else 0
agg_saa = len(all_src_correct) / len(all_verifiable) * 100 if all_verifiable else 0
hall_rate = len(all_hallucinated) / len(all_verifiable) * 100 if all_verifiable else 0

print('=' * 70)
print(f'  ACCURACY EVALUATION REPORT: {PROJECT_LABEL}')
print('=' * 70)
print(f'\n  Total properties extracted by LLM:   {len(all_results)}')
print(f'  Verifiable (Vector/OCR tagged):       {len(all_verifiable)}')
print(f'  Visual-only (manual review needed):   {len(all_visual)}')
print(f'  NOT FOUND (skipped):                  {len(all_not_found)}')
print(f'\n  --- KEY METRICS ---')
print(f'  Value Presence Rate (VPR):            {agg_vpr:.1f}%  ({len(all_verified)}/{len(all_verifiable)})')
print(f'  Source Attribution Accuracy (SAA):     {agg_saa:.1f}%  ({len(all_src_correct)}/{len(all_verifiable)})')
print(f'  Potential Hallucination Rate:          {hall_rate:.1f}%  ({len(all_hallucinated)}/{len(all_verifiable)})')
print('=' * 70)

# Per-Sheet Breakdown
print(f'\n{"Sheet":<50} {"Props":>5} {"Verif":>5} {"VPR":>6} {"SAA":>6} {"Vis":>4} {"N/F":>4} {"Hall":>4}')
print('-' * 90)
for ss in sheet_summaries:
    name = ss['sheet'][:48]
    print(f'{name:<50} {ss["total_properties"]:>5} {ss["verifiable"]:>5} {ss["vpr"]:>5.1f}% {ss["saa"]:>5.1f}% {ss["visual_only"]:>4} {ss["not_found"]:>4} {ss["hallucinated"]:>4}')

# Source Type Distribution
source_dist = defaultdict(int)
for r in all_results:
    source_dist[r['source_type']] += 1

print(f'\n-- Source Type Distribution --')
for stype, count in sorted(source_dist.items(), key=lambda x: -x[1]):
    pct = count / len(all_results) * 100
    print(f'  {stype:<20} {count:>4}  ({pct:.1f}%)')

In [ ]:
# @title 8. Failure Analysis - Properties That Failed Verification

if all_hallucinated:
    print(f'\n{len(all_hallucinated)} properties failed verification:\n')
    print(f'{"Sheet":<40} {"Property":<25} {"Value":<35} {"Claimed":>10} {"TokCov":>7} {"Fuzzy":>6}')
    print('-' * 130)
    for r in all_hallucinated:
        sheet_short = r['sheet'][:38]
        prop_short = r['property'][:23]
        val_short = r['value'][:33]
        print(f'{sheet_short:<40} {prop_short:<25} {val_short:<35} {r["source_claimed"]:>10} {r["token_coverage"]:>6.0%} {r["fuzzy_score"]:>5}')
else:
    print('\nAll verifiable properties passed - no potential hallucinations detected.')

# Source misattributions
misattributed = [r for r in all_verifiable if r['verified'] and not r['source_correct']]
if misattributed:
    print(f'\n{len(misattributed)} properties had incorrect source attribution:\n')
    for r in misattributed:
        actual_layers = []
        if r['found_in_vector']: actual_layers.append('Vector')
        if r['found_in_ocr']: actual_layers.append('OCR')
        print(f'   {r["property"]}: claimed {r["source_claimed"]}, actually in {"/".join(actual_layers)}')
else:
    print('\nAll verified properties have correct source attribution.')

In [ ]:
# @title 9. Visual-Only Properties - Export for Manual Review

if all_visual:
    print(f'\n{len(all_visual)} properties tagged Visual-only require manual verification:\n')
    print(f'{"Sheet":<40} {"Property":<30} {"Value":<50}')
    print('-' * 120)
    for r in all_visual:
        sheet_short = r['sheet'][:38]
        print(f'{sheet_short:<40} {r["property"]:<30} {r["value"]:<50}')

    manual_csv = f'{PROJECT_LABEL}_manual_review_checklist.csv'
    with open(manual_csv, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['Sheet', 'Property', 'LLM_Value', 'Correct_Yes_No', 'Corrected_Value_If_No'])
        writer.writeheader()
        for r in all_visual:
            writer.writerow({
                'Sheet': r['sheet'],
                'Property': r['property'],
                'LLM_Value': r['value'],
                'Correct_Yes_No': '',
                'Corrected_Value_If_No': ''
            })
    print(f'\nManual review checklist exported: {manual_csv}')
    print('   Fill in Correct_Yes_No (Y/N) and Corrected_Value_If_No columns.')
else:
    print('\nNo Visual-only properties - all properties have automated verification.')

In [ ]:
# @title 10. Export Full Results to CSV

# Detailed results CSV
detail_csv = f'{PROJECT_LABEL}_detailed_results.csv'
fieldnames = ['sheet', 'property', 'value', 'source_claimed', 'source_type',
              'verifiable', 'verified', 'found_in_vector', 'found_in_ocr',
              'source_correct', 'token_coverage', 'fuzzy_score', 'method', 'notes']

with open(detail_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in all_results:
        writer.writerow({k: r.get(k, '') for k in fieldnames})

print(f'Detailed results exported: {detail_csv}')

# Sheet summary CSV
summary_csv = f'{PROJECT_LABEL}_sheet_summary.csv'
with open(summary_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=list(sheet_summaries[0].keys()))
    writer.writeheader()
    writer.writerows(sheet_summaries)

print(f'Sheet summary exported: {summary_csv}')

# Aggregate metrics CSV
agg_csv = f'{PROJECT_LABEL}_aggregate_metrics.csv'
with open(agg_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Metric', 'Value', 'Count', 'Total'])
    writer.writerow(['Value Presence Rate (VPR)', f'{agg_vpr:.1f}%', len(all_verified), len(all_verifiable)])
    writer.writerow(['Source Attribution Accuracy (SAA)', f'{agg_saa:.1f}%', len(all_src_correct), len(all_verifiable)])
    writer.writerow(['Potential Hallucination Rate', f'{hall_rate:.1f}%', len(all_hallucinated), len(all_verifiable)])
    writer.writerow(['Total Properties', len(all_results), '', ''])
    writer.writerow(['Verifiable Properties', len(all_verifiable), '', ''])
    writer.writerow(['Visual-Only Properties', len(all_visual), '', ''])
    writer.writerow(['NOT FOUND Properties', len(all_not_found), '', ''])
    writer.writerow(['Sheets Evaluated', len(sheet_summaries), '', ''])

print(f'Aggregate metrics exported: {agg_csv}')
print(f'\nUse these CSVs directly in your journal paper tables.')

# Download files (Colab)
try:
    from google.colab import files
    files.download(detail_csv)
    files.download(summary_csv)
    files.download(agg_csv)
    if all_visual:
        files.download(f'{PROJECT_LABEL}_manual_review_checklist.csv')
except ImportError:
    print('Not running in Colab - files saved to current directory.')